In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from datetime import timedelta

# --- Configuration (بدون تغییر) ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output3.xlsx'

target_sensors = ['AssetID_9368','AssetID_9369','AssetID_9370','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_smart_analysis():
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return

    # 1. Data Loading & Preprocessing
    df = pd.read_excel(file_path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

    df_model = df.dropna(subset=target_sensors).copy()

    # 2. Denoising
    for sensor in target_sensors:
        df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()

    smooth_cols = [f'{s}_smooth' for s in target_sensors]
    df_model = df_model.dropna(subset=smooth_cols).copy()
    df_model = df_model.reset_index(drop=True)

    # 3. Feature Scaling
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[smooth_cols])

    # 4. Applying Local Outlier Factor (LOF)
    # n_neighbors: تعداد همسایگان برای مقایسه چگالی محلی
    # contamination: نسبت تخمینی داده‌های ناهنجار (در اینجا 0.05 یا 5 درصد فرض شده)
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

    # تشخیص وضعیت (1 برای نرمال، -1 برای ناهنجار)
    df_model['Behavior_Cluster'] = lof.fit_predict(scaled_data)

    # استخراج نمرات منفی فاکتور خارج‌افتادگی (هرچه کمتر و منفی‌تر باشد، داده ناهنجارتر است)
    # برای هماهنگی با منطق کد قبلی، آن را مثبت و معکوس می‌کنیم تا به عنوان Degradation_Index استفاده شود
    lof_scores = -lof.negative_outlier_factor_
    df_model['Degradation_Index'] = lof_scores

    # 5. Standardized Health Labeling
    # در LOF نمرات نزدیک به 1 نرمال هستند و نمرات بزرگتر (مثلاً بالای 1.5) نشان دهنده انحراف هستند
    def get_health_status(row):
        if row['Degradation_Index'] > 2.0 or row['Behavior_Cluster'] == -1:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 1.3:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

    df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)

    # 6. Filtering for the Last 30 Days
    if 'date' in df_model.columns:
        last_date = df_model['date'].max()
        final_output = df_model[df_model['date'] >= (last_date - timedelta(days=30))].copy()
    else:
        final_output = df_model

    # 7. Exporting to Excel
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print("🚀 LOF Analysis completed successfully.")
        print(f"📊 Outlier Factors calculated based on local density deviations.")

        print(f"📁 Output saved: {output_filename}")
    except Exception as e:
        print(f"❌ Error saving file: {e}")

run_smart_analysis()

🚀 LOF Analysis completed successfully.
📊 Outlier Factors calculated based on local density deviations.
📁 Output saved: outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output3.xlsx
